# Document Question Answering System (RAG)

This is my submission for the week 7 RAG assignment. In this notebook, I built a pipeline to retrieve information from custom documents and use a language model to generate grounded answers.

## Step 0: Installing Dependencies
Installing all the required packages like langchain, chroma, and transformers.

In [3]:
!pip install -q langchain-text-splitters pypdf langchain langchain-community sentence-transformers chromadb transformers rank_bm25

## Step 1: Importing Modules

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

## Step 2: Document Ingestion
I built a simple loader that accepts custom text inputs (PDFs or raw text files).

In [5]:
def ingest_document(file_path):
    print(f"Ingesting {file_path}...")
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    elif file_path.endswith('.txt'):
        loader = TextLoader(file_path)
    else:
        raise ValueError("Unsupported file type. Please provide a .pdf or .txt file.")
    documents = loader.load()
    print(f"Loaded {len(documents)} pages/sections.")
    return documents

# Creating a sample text file just for demonstration if I don't provide my own PDF
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that enhances language models by connecting them to external databases.
It retrieves relevant information from a knowledge base and appends it to the user's prompt, allowing the language model to generate factually accurate answers.
Vector databases are commonly used in RAG to store document embeddings and perform fast similarity search.
Chunking is the process of breaking large documents into smaller segments to improve retrieval precision.
"""
with open('sample_doc.txt', 'w') as f:
    f.write(sample_text)

documents = ingest_document('sample_doc.txt')

Ingesting sample_doc.txt...
Loaded 1 pages/sections.


## Step 3: Text Chunking
Processing the raw text by breaking it into smaller chunks. I went with a chunk size of 200 with an overlap of 50 after experimenting a bit to keep the context intact.

In [6]:
# Experimenting with chunking profiles
chunk_size = 200
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""]
)
chunked_docs = text_splitter.split_documents(documents)
print(f"Split document into {len(chunked_docs)} chunks.")
for i, chunk in enumerate(chunked_docs[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content}")

Split document into 4 chunks.

Chunk 1:
Retrieval-Augmented Generation (RAG) is a technique that enhances language models by connecting them to external databases.

Chunk 2:
It retrieves relevant information from a knowledge base and appends it to the user's prompt, allowing the language model to generate factually accurate answers.

Chunk 3:
Vector databases are commonly used in RAG to store document embeddings and perform fast similarity search.


## Step 4: Embeddings and Vector Database
Here I'm mapping the chunked strings into vector representations using a pre-trained model from HuggingFace, and then storing them in ChromaDB for fast similarity search.

In [7]:
# Using a small embedding model from HuggingFace
embedding_model_name = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Initialize Vector Database
vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("Vector store initialized and embeddings stored.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store initialized and embeddings stored.


## Step 5 & 6: Querying and Retrieval
Creating a route that takes an incoming question and queries the vector store to extract the most relevant document chunks.

In [9]:
def retrieve_context(query, k=2):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    relevant_docs = retriever.invoke(query)
    return relevant_docs

test_query = "What is the purpose of chunking?"
retrieved_docs = retrieve_context(test_query)

print(f"Query: {test_query}")
print("\nRetrieved Context:")
for doc in retrieved_docs:
    print(f"- {doc.page_content}")

Query: What is the purpose of chunking?

Retrieved Context:
- Chunking is the process of breaking large documents into smaller segments to improve retrieval precision.
- It retrieves relevant information from a knowledge base and appends it to the user's prompt, allowing the language model to generate factually accurate answers.


## Step 7: Language Model and Generation
Connecting the retrieved context and the query into a unified prompt. I decided to use `TinyLlama` as my LLM because it's a great lightweight model for text generation that I can run entirely locally.

In [12]:
# Using a small local model for demonstration purposes to avoid API key requirements.
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading Language Model: {model_id}...")

llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    max_new_tokens=256,
    temperature=0.1
)

local_llm = HuggingFacePipeline(pipeline=llm_pipeline)

# Unified Prompt Template
prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}
Answer:"""
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

# Setting up the retrieval pipeline (LCEL)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

qa_chain = (
    {'context': retriever, 'question': RunnablePassthrough()} 
    | PROMPT 
    | local_llm
)






print("QA Pipeline ready.")

Loading Language Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


QA Pipeline ready.


## Step 8: Validation Logs
Testing the end-to-end pipeline with some sample questions to see how well it extracts text and generates answers.

In [13]:
queries = [
    "What is Retrieval-Augmented Generation?",
    "Why do we use vector databases in RAG?",
    "What is chunking?"
]

print("--- Testing the Pipeline ---\n")
for q in queries:
    print(f"\n[Question]: {q}")
    result = qa_chain.invoke(q)
    print(f"[Answer]: {result}")
    print("[Source Contexts]:")
    docs = retriever.invoke(q)
    for doc in docs:
        print(f"   -> {doc.page_content.replace(chr(10), ' ')}")

--- VALIDATION LOGS ---

[Question]: What is Retrieval-Augmented Generation?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[Answer]: Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: [Document(metadata={'source': 'sample_doc.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a technique that enhances language models by connecting them to external databases.'), Document(metadata={'source': 'sample_doc.txt'}, page_content="It retrieves relevant information from a knowledge base and appends it to the user's prompt, allowing the language model to generate factually accurate answers.")]

Question: What is Retrieval-Augmented Generation?
Answer:Retrieval-Augmented Generation (RAG) is a technique that enhances language models by connecting them to external databases.
[Source Contexts]:
   -> Retrieval-Augmented Generation (RAG) is a technique that enhances language models by connecting them to external databases.
   -> It retrieves relevant information from a knowledge base and appe

## Step 9: System Metrics & Project Summary

### System Metrics Report
- **Chunking**: `RecursiveCharacterTextSplitter` (size 200, overlap 50). I found this size works well for precise retrieval.
- **Embeddings**: `all-MiniLM-L6-v2` (384 dimensions). It's fast and accurate.
- **Vector Store**: `ChromaDB` for local storage.
- **Language Model**: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.

### Experiments & Optimizations
For further optimization, I looked into **Hybrid Search**. This means combining vector search with basic keyword search (like BM25). This would help catch exact keyword matches that vector embeddings sometimes miss. Adding a re-ranking model layer on top of that would also improve things.

### Key Learnings
- How RAG systems combine retrieval and generation.
- The importance of retrieval in improving answer accuracy.
- Working with embeddings and vector databases.
- Handling unstructured text data.
- Designing scalable AI pipelines.

### Conclusion
This project demonstrates how to build a system that can understand user queries, retrieve relevant information, and generate accurate answers. RAG systems like this are widely used in chatbots, knowledge assistants, enterprise search systems, and AI-powered documentation tools.